# LightGBM-XGBoost Ensemble Challenge

This experiment asks whether XGBoost adds complementary ranking signal to the current LightGBM reference. Both models receive the same 467 leakage-safe features. Blend weights are selected only from three expanding TRAIN temporal OOF folds; external VALIDATION is accessed only if a blend strictly wins, and HOLDOUT is never assigned or accessed.

In [1]:
import gc
import json
import sys
import time
from pathlib import Path

import lightgbm as lgb
import mlflow
import numpy as np
import pandas as pd
import xgboost as xgb
from IPython.display import display
from sklearn.metrics import average_precision_score, roc_auc_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fraud_detection.data import (
    load_train_identity,
    load_train_transaction,
    merge_train_data,
    split_data_chronologically,
    validate_training_data,
)
from fraud_detection.features import COMPOSITE_TARGET_ENTITIES
from fraud_detection.models import (
    aggregate_oof_predictions,
    evaluate_probabilities,
    evaluate_probability_blends,
    make_expanding_time_folds,
    prediction_correlations,
    prepare_engineered_matrices,
    summarize_disagreement,
    weighted_score_blend,
)

notebook_started = time.perf_counter()

2026/10/08 10:56:19 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\fraud-detection-system\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Could not save font_manager cache [Errno 13] Permission denied: 'C:\\Users\\Marwan\\.matplotlib\\fontlist-v3.11.0.json.matplotlib-lock'


## Fixed design

LightGBM reuses the exact configuration behind the 0.612204 external reference. XGBoost reuses the earlier model-comparison winner and its fit-period class ratio. There is no tuning.

In [2]:
lightgbm_parameters = {
    "learning_rate": 0.1,
    "n_estimators": 400,
    "num_leaves": 63,
    "class_weight": "balanced",
    "random_state": 42,
    "n_jobs": -1,
    "verbosity": -1,
}
xgboost_parameters = {
    "learning_rate": 0.1,
    "max_depth": 7,
    "n_estimators": 300,
    "tree_method": "hist",
    "eval_metric": "aucpr",
    "random_state": 42,
    "n_jobs": -1,
}
composite_te_features = tuple(
    f"{entity}_target_encoded_fraud_rate" for entity in COMPOSITE_TARGET_ENTITIES
)
blend_weights = (1.0, 0.9, 0.8, 0.7, 0.6, 0.5)
tracking_uri = f"sqlite:///{(PROJECT_ROOT / 'model_comparison.db').resolve().as_posix()}"
experiment_name = "fraud-detection-ensemble-challenge"
mlflow.set_tracking_uri(tracking_uri)
mlflow.set_experiment(experiment_name)
print("LightGBM:", lightgbm_parameters)
print("XGBoost:", xgboost_parameters, "plus fit-period scale_pos_weight")

2026/10/08 10:56:21 INFO mlflow.tracking.fluent: Experiment with name 'fraud-detection-ensemble-challenge' does not exist. Creating a new experiment.


LightGBM: {'learning_rate': 0.1, 'n_estimators': 400, 'num_leaves': 63, 'class_weight': 'balanced', 'random_state': 42, 'n_jobs': -1, 'verbosity': -1}
XGBoost: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 300, 'tree_method': 'hist', 'eval_metric': 'aucpr', 'random_state': 42, 'n_jobs': -1} plus fit-period scale_pos_weight


## Leakage-safe temporal OOF predictions

Each fold rebuilds frequency maps, historical amount state, composites, temporal OOF target encodings, frozen evaluation maps, imputations, and scaling from its fit history. Evaluation targets are used only after predictions exist.

In [3]:
raw_dir = PROJECT_ROOT / "data" / "raw"
transaction = load_train_transaction(raw_dir)
identity = load_train_identity(raw_dir)
merged = merge_train_data(transaction, identity)
split = split_data_chronologically(merged)
train = split.train
validate_training_data(train)
del transaction, identity, merged
gc.collect()

folds = make_expanding_time_folds(train, n_splits=3, initial_train_fraction=0.70)
fold_design = pd.DataFrame(
    [
        {
            "fold": number,
            "fit_rows": len(fold.train_indices),
            "evaluation_rows": len(fold.evaluation_indices),
            "fit_dt_max": train.iloc[fold.train_indices]["TransactionDT"].max(),
            "evaluation_dt_min": train.iloc[fold.evaluation_indices]["TransactionDT"].min(),
            "evaluation_dt_max": train.iloc[fold.evaluation_indices]["TransactionDT"].max(),
        }
        for number, fold in enumerate(folds, start=1)
    ]
)
display(fold_design)
print("External validation status: not assigned or accessed before OOF selection")
print("HOLDOUT status: not assigned or accessed")

,fold,fit_rows,evaluation_rows,fit_dt_max,evaluation_dt_min,evaluation_dt_max
0,1,289364,41338,7135483,7135609,8121450
1,2,330702,41338,8121450,8121466,9247568
2,3,372040,41338,9247568,9247583,10437996


External validation status: not assigned or accessed before OOF selection
HOLDOUT status: not assigned or accessed


In [4]:
lightgbm_oof_parts = []
xgboost_oof_parts = []
preparation_seconds = []
lightgbm_seconds = []
xgboost_seconds = []
prepared_memory_mib = []
fold_scale_pos_weights = []
feature_count = None

for fold_number, fold in enumerate(folds, start=1):
    fit_frame = train.iloc[fold.train_indices]
    evaluation_frame = train.iloc[fold.evaluation_indices]
    preparation_started = time.perf_counter()
    prepared = prepare_engineered_matrices(
        fit_frame,
        evaluation_frame,
        include_graph=False,
        include_composite=True,
        include_target_encoding=True,
        scaling_batch_rows=len(fit_frame),
    )
    preparation_seconds.append(time.perf_counter() - preparation_started)
    prepared_memory_mib.append((prepared.x_train.nbytes + prepared.x_evaluation.nbytes) / 1024**2)
    positions = {name: index for index, name in enumerate(prepared.feature_names)}
    baseline_names = tuple(
        name for name in prepared.feature_names if not name.endswith("_target_encoded_fraud_rate")
    )
    selected_names = baseline_names + composite_te_features
    selected = np.asarray([positions[name] for name in selected_names], dtype=np.int32)
    feature_count = len(selected)
    if feature_count != 467:
        raise RuntimeError(f"Expected 467 features, received {feature_count}")
    x_fit = prepared.x_train[:, selected]
    x_evaluation = prepared.x_evaluation[:, selected]
    y_fit = fit_frame["isFraud"].astype("int8")
    y_evaluation = evaluation_frame["isFraud"].astype("int8")

    started = time.perf_counter()
    lightgbm_model = lgb.LGBMClassifier(**lightgbm_parameters)
    lightgbm_model.fit(x_fit, y_fit)
    lightgbm_score = lightgbm_model.predict_proba(x_evaluation)[:, 1]
    lightgbm_seconds.append(time.perf_counter() - started)

    scale_pos_weight = float((y_fit == 0).sum() / (y_fit == 1).sum())
    fold_scale_pos_weights.append(scale_pos_weight)
    started = time.perf_counter()
    xgboost_model = xgb.XGBClassifier(**xgboost_parameters, scale_pos_weight=scale_pos_weight)
    xgboost_model.fit(x_fit, y_fit)
    xgboost_score = xgboost_model.predict_proba(x_evaluation)[:, 1]
    xgboost_seconds.append(time.perf_counter() - started)

    keys = pd.DataFrame(
        {
            "TransactionID": evaluation_frame["TransactionID"].to_numpy(),
            "fold": fold_number,
            "y_true": y_evaluation.to_numpy(),
        }
    )
    lightgbm_oof_parts.append(keys.assign(score=lightgbm_score))
    xgboost_oof_parts.append(keys.assign(score=xgboost_score))
    print(
        f"fold={fold_number} features={feature_count} "
        f"lgbm_pr_auc={average_precision_score(y_evaluation, lightgbm_score):.6f} "
        f"xgb_pr_auc={average_precision_score(y_evaluation, xgboost_score):.6f}"
    )
    del prepared, x_fit, x_evaluation, lightgbm_model, xgboost_model
    gc.collect()

oof_predictions = aggregate_oof_predictions(
    pd.concat(lightgbm_oof_parts, ignore_index=True),
    pd.concat(xgboost_oof_parts, ignore_index=True),
)
oof_path = PROJECT_ROOT / "reports" / "ensemble_oof_predictions.csv"
oof_predictions.to_csv(oof_path, index=False)
print(f"Persisted {len(oof_predictions):,} aligned OOF rows to {oof_path.name}")

fold=1 features=467 lgbm_pr_auc=0.596947 xgb_pr_auc=0.579794


fold=2 features=467 lgbm_pr_auc=0.576060 xgb_pr_auc=0.550805


fold=3 features=467 lgbm_pr_auc=0.630908 xgb_pr_auc=0.605440


Persisted 124,014 aligned OOF rows to ensemble_oof_predictions.csv


## Correlation, disagreement, and fixed blend search

A material rank disagreement means an absolute difference of at least 20 percentile points in overall OOF score rank. A LightGBM-missed fraud is a positive OOF row below the unoptimized 0.50 threshold.

In [5]:
correlations = prediction_correlations(oof_predictions)
disagreement = summarize_disagreement(oof_predictions, material_rank_difference=0.20)
blend_results = evaluate_probability_blends(oof_predictions, blend_weights)


def model_cv_rows(frame):
    rows = []
    for model_name, score_column in (("LightGBM", "lightgbm_score"), ("XGBoost", "xgboost_score")):
        fold_pr = []
        fold_roc = []
        for _, fold_frame in frame.groupby("fold", sort=True):
            fold_pr.append(average_precision_score(fold_frame["y_true"], fold_frame[score_column]))
            fold_roc.append(roc_auc_score(fold_frame["y_true"], fold_frame[score_column]))
        rows.append(
            {
                "model": model_name,
                "fold_pr_auc": tuple(float(value) for value in fold_pr),
                "mean_pr_auc": float(np.mean(fold_pr)),
                "std_pr_auc": float(np.std(fold_pr, ddof=0)),
                "mean_roc_auc": float(np.mean(fold_roc)),
            }
        )
    return pd.DataFrame(rows)


model_results = model_cv_rows(oof_predictions)
overall = correlations.loc[correlations["fold"] == "overall"].iloc[0]
display(correlations)
display(model_results)
display(blend_results)
display(
    pd.Series(
        disagreement.__dict__
        if hasattr(disagreement, "__dict__")
        else {field: getattr(disagreement, field) for field in disagreement.__dataclass_fields__}
    )
)

,fold,row_count,pearson,spearman
0,1,41338,0.892714,0.881451
1,2,41338,0.913881,0.898374
2,3,41338,0.932636,0.927224
3,overall,124014,0.916229,0.900851


,model,fold_pr_auc,mean_pr_auc,std_pr_auc,mean_roc_auc
0,LightGBM,"(0.5969474918067366, 0.5760601358100342, 0.630...",0.601305,0.022603,0.890932
1,XGBoost,"(0.5797937833064107, 0.5508054544894174, 0.605...",0.578680,0.022318,0.884211


,lightgbm_weight,xgboost_weight,fold_pr_auc,fold_roc_auc,mean_pr_auc,std_pr_auc,mean_roc_auc
0,0.8,0.2,"(0.6022350315687843, 0.5793011711245719, 0.631...","(0.8780342020625803, 0.897137007468736, 0.9045...",0.604372,0.021396,0.893230
1,0.7,0.3,"(0.6031441026154423, 0.5790605239895477, 0.630...","(0.8777311703310713, 0.8969725112157788, 0.904...",0.604333,0.021137,0.893203
2,0.9,0.1,"(0.6002212475013049, 0.5788674324980717, 0.631...","(0.877810810868722, 0.8967788940285933, 0.9035...",0.603558,0.021651,0.892699
3,0.6,0.4,"(0.603008222126439, 0.5775940540456028, 0.6292...","(0.8771476311182629, 0.8964265496676735, 0.904...",0.603288,0.021094,0.892827
4,0.5,0.5,"(0.6019324836823109, 0.5754029108130742, 0.627...","(0.8763329012185637, 0.895555573695148, 0.9046...",0.601504,0.021139,0.892175
5,1.0,0.0,"(0.5969474918067366, 0.5760601358100342, 0.630...","(0.8763734585823351, 0.8954762531611109, 0.900...",0.601305,0.022603,0.890932


material_rank_difference_threshold                0.200000
materially_different_rank_fraction                0.119180
lightgbm_missed_fraud_count                    2073.000000
mean_xgboost_minus_lightgbm_on_missed_fraud       0.049968
xgboost_higher_on_missed_fraud_fraction           0.728413
xgboost_rescue_at_0_50_fraction                   0.082489
dtype: float64

In [6]:
run_ids = {}
common_tags = {
    "stage": "ensemble-challenge",
    "dataset": "ieee-cis",
    "selection": "temporal-oof",
}
for _, row in model_results.iterrows():
    with mlflow.start_run(run_name=f"oof-{row['model'].lower()}") as run:
        mlflow.set_tags(common_tags)
        mlflow.log_params({"model_name": row["model"], "feature_count": feature_count})
        metrics = {
            "mean_cv_pr_auc": row["mean_pr_auc"],
            "std_cv_pr_auc": row["std_pr_auc"],
            "mean_cv_roc_auc": row["mean_roc_auc"],
            "pearson_correlation": overall["pearson"],
            "spearman_correlation": overall["spearman"],
        }
        metrics.update({f"fold_{i}_pr_auc": value for i, value in enumerate(row["fold_pr_auc"], 1)})
        mlflow.log_metrics(metrics)
        run_ids[f"model_{row['model']}"] = run.info.run_id

for _, row in blend_results.sort_values("lightgbm_weight", ascending=False).iterrows():
    run_name = f"blend-lgbm-{row['lightgbm_weight']:.1f}-xgb-{row['xgboost_weight']:.1f}"
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags(common_tags)
        mlflow.log_params(
            {
                "lightgbm_weight": row["lightgbm_weight"],
                "xgboost_weight": row["xgboost_weight"],
                "feature_count": feature_count,
            }
        )
        metrics = {
            "mean_cv_pr_auc": row["mean_pr_auc"],
            "std_cv_pr_auc": row["std_pr_auc"],
            "mean_cv_roc_auc": row["mean_roc_auc"],
            "pearson_correlation": overall["pearson"],
            "spearman_correlation": overall["spearman"],
        }
        metrics.update({f"fold_{i}_pr_auc": value for i, value in enumerate(row["fold_pr_auc"], 1)})
        mlflow.log_metrics(metrics)
        run_ids[run_name] = run.info.run_id

best_blend = blend_results.iloc[0]
lightgbm_alone = blend_results.loc[blend_results["lightgbm_weight"] == 1.0].iloc[0]
ensemble_selected = bool(
    best_blend["xgboost_weight"] > 0 and best_blend["mean_pr_auc"] > lightgbm_alone["mean_pr_auc"]
)
absolute_cv_gain = float(best_blend["mean_pr_auc"] - lightgbm_alone["mean_pr_auc"])
relative_cv_gain = absolute_cv_gain / float(lightgbm_alone["mean_pr_auc"])
print(
    f"OOF winner: {best_blend['lightgbm_weight']:.1f}/{best_blend['xgboost_weight']:.1f}; "
    f"gain={absolute_cv_gain:+.6f} ({relative_cv_gain:+.2%})"
)
print(f"Ensemble selected: {ensemble_selected}")

OOF winner: 0.8/0.2; gain=+0.003066 (+0.51%)
Ensemble selected: True


## Conditional external validation

The following branch executes exactly once only when TRAIN OOF selects a blend. The selected weight is frozen before VALIDATION is accessed. A losing ensemble is not evaluated externally.

In [7]:
validation_metrics = None
final_seconds = 0.0
final_memory_mib = 0.0
if ensemble_selected:
    validation = split.validation
    final_started = time.perf_counter()
    prepared_final = prepare_engineered_matrices(
        train,
        validation,
        include_graph=False,
        include_composite=True,
        include_target_encoding=True,
        scaling_batch_rows=len(train),
    )
    positions = {name: index for index, name in enumerate(prepared_final.feature_names)}
    baseline_names = tuple(
        name
        for name in prepared_final.feature_names
        if not name.endswith("_target_encoded_fraud_rate")
    )
    selected_names = baseline_names + composite_te_features
    selected = np.asarray([positions[name] for name in selected_names], dtype=np.int32)
    x_fit = prepared_final.x_train[:, selected]
    x_validation = prepared_final.x_evaluation[:, selected]
    y_fit = train["isFraud"].astype("int8")

    final_lightgbm = lgb.LGBMClassifier(**lightgbm_parameters)
    final_lightgbm.fit(x_fit, y_fit)
    lightgbm_validation_score = final_lightgbm.predict_proba(x_validation)[:, 1]
    full_scale_pos_weight = float((y_fit == 0).sum() / (y_fit == 1).sum())
    final_xgboost = xgb.XGBClassifier(**xgboost_parameters, scale_pos_weight=full_scale_pos_weight)
    final_xgboost.fit(x_fit, y_fit)
    xgboost_validation_score = final_xgboost.predict_proba(x_validation)[:, 1]
    validation_score = weighted_score_blend(
        lightgbm_validation_score,
        xgboost_validation_score,
        lightgbm_weight=float(best_blend["lightgbm_weight"]),
        xgboost_weight=float(best_blend["xgboost_weight"]),
    )
    validation_metrics = evaluate_probabilities(validation["isFraud"], validation_score)
    final_seconds = time.perf_counter() - final_started
    final_memory_mib = (
        prepared_final.x_train.nbytes + prepared_final.x_evaluation.nbytes
    ) / 1024**2
    selected_run_name = (
        f"blend-lgbm-{best_blend['lightgbm_weight']:.1f}-xgb-{best_blend['xgboost_weight']:.1f}"
    )
    with mlflow.start_run(run_id=run_ids[selected_run_name]):
        mlflow.set_tag("external_validation_evaluated", "true")
        mlflow.log_metrics(
            {
                "validation_pr_auc": validation_metrics.pr_auc,
                "validation_roc_auc": validation_metrics.roc_auc,
                "validation_precision_at_0_50": validation_metrics.precision,
                "validation_recall_at_0_50": validation_metrics.recall,
                "validation_f1_at_0_50": validation_metrics.f1,
            }
        )
    print(validation_metrics)
else:
    print("LightGBM won TRAIN OOF; external VALIDATION was not accessed or re-evaluated.")

BinaryMetrics(pr_auc=0.6152645019628108, roc_auc=0.9162239563163297, precision=0.5887881839543471, recall=0.576594345825115, f1=0.5826274705198472, confusion_matrix=((84314, 1225), (1288, 1754)))


## Report and recommendation

In [8]:
def fold_values(values):
    return " / ".join(f"{value:.6f}" for value in values)


model_lines = [
    "| Model | Fold PR-AUCs | Mean PR-AUC | Std PR-AUC | Mean ROC-AUC |",
    "| --- | --- | ---: | ---: | ---: |",
]
for _, row in model_results.iterrows():
    model_lines.append(
        f"| {row['model']} | {fold_values(row['fold_pr_auc'])} | "
        f"{row['mean_pr_auc']:.6f} | {row['std_pr_auc']:.6f} | "
        f"{row['mean_roc_auc']:.6f} |"
    )
blend_lines = [
    "| LightGBM / XGBoost | Fold PR-AUCs | Mean PR-AUC | Std PR-AUC | Mean ROC-AUC |",
    "| --- | --- | ---: | ---: | ---: |",
]
for _, row in blend_results.sort_values("lightgbm_weight", ascending=False).iterrows():
    blend_lines.append(
        f"| {row['lightgbm_weight']:.1f} / {row['xgboost_weight']:.1f} | "
        f"{fold_values(row['fold_pr_auc'])} | {row['mean_pr_auc']:.6f} | "
        f"{row['std_pr_auc']:.6f} | {row['mean_roc_auc']:.6f} |"
    )
correlation_lines = [
    "| Fold | Rows | Pearson | Spearman |",
    "| --- | ---: | ---: | ---: |",
]
for _, row in correlations.iterrows():
    correlation_lines.append(
        f"| {row['fold']} | {int(row['row_count'])} | "
        f"{row['pearson']:.6f} | {row['spearman']:.6f} |"
    )
fold_lines = "\n".join(
    f"- Fold {int(row['fold'])}: fit {int(row['fit_rows']):,}, "
    f"evaluate {int(row['evaluation_rows']):,}, DT after {row['fit_dt_max']} "
    f"through {row['evaluation_dt_max']}"
    for _, row in fold_design.iterrows()
)
reference_difference = (validation_metrics.pr_auc - 0.612204) if validation_metrics else None
missed_fraud_delta = disagreement.mean_xgboost_minus_lightgbm_on_missed_fraud
selected_weight_text = f"{best_blend['lightgbm_weight']:.1f} / {best_blend['xgboost_weight']:.1f}"
validation_section = (
    f"""- PR-AUC: **{validation_metrics.pr_auc:.6f}**
- ROC-AUC: **{validation_metrics.roc_auc:.6f}**
- Precision at 0.50: **{validation_metrics.precision:.6f}**
- Recall at 0.50: **{validation_metrics.recall:.6f}**
- F1 at 0.50: **{validation_metrics.f1:.6f}**
- Confusion matrix: **{validation_metrics.confusion_matrix}**
- Difference from LightGBM external reference 0.612204: **{reference_difference:+.6f}**"""
    if validation_metrics is not None
    else "Not run: no ensemble beat LightGBM on mean TRAIN OOF PR-AUC."
)
total_seconds = time.perf_counter() - notebook_started
peak_matrix_mib = max([*prepared_memory_mib, final_memory_mib])
material = absolute_cv_gain >= 0.001
recommendation = (
    "Use the selected ensemble: its OOF gain is at least 0.001 PR-AUC."
    if ensemble_selected and material
    else "Keep LightGBM alone: the added XGBoost serving path is not justified by the OOF result."
)
run_lines = "\n".join(f"- **{name}:** `{run_id}`" for name, run_id in run_ids.items())
report = f"""# LightGBM-XGBoost Ensemble Challenge

## Component models and feature pipeline

Both models use the same **{feature_count}** predictors: existing frequency encodings, five
past-only historical amount features, 20 composite identity features, and four composite
temporal target-encoded rates. Graph features and time-decay weighting are excluded.

- LightGBM: `{json.dumps(lightgbm_parameters, sort_keys=True)}`
- XGBoost: `{json.dumps(xgboost_parameters, sort_keys=True)}`, plus fit-period
  `scale_pos_weight = non_fraud / fraud` (fold values: {fold_values(fold_scale_pos_weights)}).

## Temporal OOF methodology

{fold_lines}

All preprocessing was rebuilt from each fold's historical fit rows. Evaluation labels were used
only for scoring. The aligned OOF artifact contains {len(oof_predictions):,} rows.

## Individual model CV metrics

{chr(10).join(model_lines)}

## Prediction correlation

{chr(10).join(correlation_lines)}

- Material rank-difference threshold: **{disagreement.material_rank_difference_threshold:.0%}**
- Rows materially different: **{disagreement.materially_different_rank_fraction:.2%}**
- Fraud rows below LightGBM 0.50: **{disagreement.lightgbm_missed_fraud_count:,}**
- Mean XGBoost minus LightGBM score on those rows: **{missed_fraud_delta:+.6f}**
- XGBoost score higher on those rows: **{disagreement.xgboost_higher_on_missed_fraud_fraction:.2%}**
- XGBoost score at least 0.50 on those rows: **{disagreement.xgboost_rescue_at_0_50_fraction:.2%}**

## Probability-blend search

{chr(10).join(blend_lines)}

- Selected OOF candidate: **{selected_weight_text}**
- Absolute gain over LightGBM: **{absolute_cv_gain:+.6f}**
- Relative gain over LightGBM: **{relative_cv_gain:+.2%}**
- Ensemble strictly beat LightGBM: **{ensemble_selected}**

## External validation

{validation_section}

The existing LightGBM external reference is PR-AUC **0.612204**, ROC-AUC **0.914085**,
precision **0.600277**, recall **0.569691**, and F1 **0.584584**. These values did not
participate in OOF blend selection.

## Runtime, memory, and serving trade-off

- Fold preparation seconds: **{fold_values(preparation_seconds)}**
- LightGBM fit/predict seconds: **{fold_values(lightgbm_seconds)}**
- XGBoost fit/predict seconds: **{fold_values(xgboost_seconds)}**
- Conditional final TRAIN/VALIDATION seconds: **{final_seconds:.1f}**
- Peak prepared-matrix estimate: **{peak_matrix_mib:.1f} MiB**
- Total notebook runtime before report write: **{total_seconds:.1f} seconds**

An ensemble requires storing, loading, scoring, monitoring, and versioning two models. This
roughly doubles booster inference work and adds blend coordination.

**Recommendation:** {recommendation}

## MLflow

- Experiment: `{experiment_name}`
- Tracking URI: `{tracking_uri}`
{run_lines}

## Leakage boundary

Ensemble weights were selected only from TRAIN temporal OOF predictions. External VALIDATION
did not influence selection and was evaluated only if an ensemble won. HOLDOUT was never
assigned, inspected, transformed, predicted, summarized, or evaluated.
"""
report_path = PROJECT_ROOT / "reports" / "ensemble_challenge_summary.md"
report_path.write_text(report, encoding="utf-8")
print(f"Report: {report_path.relative_to(PROJECT_ROOT)}")
print(f"Total runtime: {total_seconds:.1f} seconds")
print(f"Recommendation: {recommendation}")
print("External VALIDATION influenced selection: False")
print("HOLDOUT status: untouched")

Report: reports\ensemble_challenge_summary.md
Total runtime: 547.8 seconds
Recommendation: Use the selected ensemble: its OOF gain is at least 0.001 PR-AUC.
External VALIDATION influenced selection: False
HOLDOUT status: untouched
